https://www.youtube.com/watch?v=pkVwUVEHmfI

In [2]:
import numpy as np

### Example 1

In [69]:
A = np.random.rand(3, 5)
B = np.random.rand(5, 2)

print(A.shape)
print(B.shape)


M1 = np.empty((3, 2))

(3, 5)
(5, 2)


In [70]:
for i in range(3):
  for j in range(2):
    total = 0
    for k in range(5):
      total += A[i, k] * B[k, j]
    M1[i, j] = total
M1

array([[1.37370657, 1.28153183],
       [0.79552501, 1.20957685],
       [1.07451342, 1.26271296]])

In [71]:
M2 = np.empty((3, 2))
M2 = np.einsum('ik, kj -> ij', A, B)
M1==M2

array([[ True,  True],
       [ True,  True],
       [ True,  True]])

Free indices: i, j
Summation index: k

### Example 2

In [72]:
a = np.random.rand(5)
print(a.shape)
a = np.array([1, 2, 3, 4, 5])
print(a.shape)

# b = np.random.rand(3)
b = np.array([1, 2, 3])
outer1 = np.empty((5, 3))

outer1 = np.einsum('i,j->ij', a, b)
outer1

(5,)
(5,)


array([[ 1,  2,  3],
       [ 2,  4,  6],
       [ 3,  6,  9],
       [ 4,  8, 12],
       [ 5, 10, 15]])

In [73]:
outer2 = np.empty((5, 3))
for i in range(5):
  for j in range(3):
    total = 0

    total += a[i] * b[j]
    outer2[i, j] = total
outer2

array([[ 1.,  2.,  3.],
       [ 2.,  4.,  6.],
       [ 3.,  6.,  9.],
       [ 4.,  8., 12.],
       [ 5., 10., 15.]])

In [74]:
outer1 == outer2

array([[ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True]])

### Rules:
1. Repeating letters in different inputs means those values will be multiplied and those products will be the output. M2 = np.einsum('ik, kj -> ij', A, B)
2. Omitting a letter means that axis will be summed
3. We can retun the unsummed axis in any order

In [75]:
# 2
X = np.ones(3)
sum_X = np.einsum('i->', X)
sum_X

np.float64(3.0)

In [76]:
# 3
X = np.ones((5, 4, 3))
X_modified = np.einsum('ijk->kji', X)
X_modified.shape

(3, 4, 5)

In [77]:
import torch

In [78]:
def printShape(x: torch.Tensor):
  print(x.shape)
  print(x)

In [79]:
x = torch.rand((2, 3))

printShape(x)

torch.Size([2, 3])
tensor([[0.3203, 0.3099, 0.3603],
        [0.9781, 0.4562, 0.6035]])


In [80]:
# Permutation of Tensors / Transpose
x1 = torch.einsum('ij->ji', x)
printShape(x1)

torch.Size([3, 2])
tensor([[0.3203, 0.9781],
        [0.3099, 0.4562],
        [0.3603, 0.6035]])


In [81]:
# Summation
x2 = torch.einsum('ij->', x)
printShape(x2)

torch.Size([])
tensor(3.0283)


In [82]:
# Column Sum
x3 = torch.einsum('ij->j',x)
printShape(x3)

torch.Size([3])
tensor([1.2984, 0.7661, 0.9638])


In [83]:
# Row Sum
x4 = torch.einsum('ij->i', x)
printShape(x4)

torch.Size([2])
tensor([0.9905, 2.0378])


In [84]:
# Matrix-Vector Multiplication
v = torch.rand((1, 3))
x5 = torch.einsum("ij,kj->ik", x, v)
printShape(x5)

torch.Size([2, 1])
tensor([[0.8939],
        [1.8794]])


In [89]:
# Matrix - Matrix Multiplication
x6 = torch.einsum("ij,kj->ik", x, x) # (2, 2) : (2, 3) X (3, 2)
printShape(x6)

torch.Size([2, 2])
tensor([[0.3285, 0.6721],
        [0.6721, 1.5290]])


In [91]:
# Dot product first row with first row of matrix
x7 = torch.einsum("i,i->", x[0], x[0])
printShape(x7)

torch.Size([])
tensor(0.3285)


In [92]:
# Dot product with matrix
x8 = torch.einsum("ij,ij->", x, x)
printShape(x8)

torch.Size([])
tensor(1.8575)


In [96]:
# Hadamard Product (element-wise multiplication)
x9 = torch.einsum("ij,ij->ij", x, x)
printShape(x9)

torch.Size([2, 3])
tensor([[0.1026, 0.0961, 0.1298],
        [0.9566, 0.2081, 0.3642]])


In [95]:
# Outer Product
a = torch.rand((3))
b = torch.rand((5))

x10 = torch.einsum("i,j->ij", a, b)
printShape(x10)

torch.Size([3, 5])
tensor([[0.0069, 0.0101, 0.0037, 0.0064, 0.0147],
        [0.1696, 0.2499, 0.0921, 0.1577, 0.3646],
        [0.2241, 0.3302, 0.1217, 0.2083, 0.4817]])


In [99]:
# Batch Matrix Multiplication
c = torch.rand((3, 2, 5))
d = torch.rand((3, 5, 3))

x11 = torch.einsum("ijk,ikl->ijl", c, d)
printShape(x11)

torch.Size([3, 2, 3])
tensor([[[0.5720, 1.1457, 1.3126],
         [1.1232, 2.4329, 2.2947]],

        [[1.7381, 1.0863, 1.1637],
         [1.6928, 1.2845, 1.1687]],

        [[1.0777, 1.5646, 1.6635],
         [0.9016, 1.4441, 1.2783]]])


In [102]:
# Matrix Diagonal
x = torch.rand((3, 3))
x12 = torch.einsum("ii->i", x)
printShape(x12)

torch.Size([3])
tensor([0.0259, 0.3244, 0.5391])


In [103]:
# Matrix Trace
x13 = torch.einsum("ii->", x)
printShape(x13)

torch.Size([])
tensor(0.8895)
